In [10]:
import shutil
import os

# Folder path to delete
folder_path = r"/mnt/lustre/lalit47/down/class/MMOTU"

# Check if folder exists
if os.path.exists(folder_path):

    # Delete folder and all contents
    shutil.rmtree(folder_path)

    print(f"Deleted folder: {folder_path}")

else:
    print("Folder does not exist")

Deleted folder: /mnt/lustre/lalit47/down/class/MMOTU


In [9]:
import os
import shutil
import cv2
# =========================
# USER INPUT PATHS
# =========================

# Source dataset path
source_path = r"/mnt/lustre/lalit47/us_data/bench/OTU_2d"

# Target output path
target_path = r"/mnt/lustre/lalit47/down/seg/MMOTU"

# Text files
train_txt = os.path.join(source_path, "train.txt")
val_txt = os.path.join(source_path, "val.txt")

# Source folders
images_folder = os.path.join(source_path, "images")
masks_folder = os.path.join(source_path, "annotations")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

# Create target directories
os.makedirs(train_images_out, exist_ok=True)
os.makedirs(train_masks_out, exist_ok=True)

os.makedirs(val_images_out, exist_ok=True)
os.makedirs(val_masks_out, exist_ok=True)


def process_and_save(src, dst):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)


def copy_dataset(txt_file, img_output, mask_output):
    """
    Process images and masks listed in txt_file
    """

    with open(txt_file, "r") as f:
        file_names = [line.strip() for line in f.readlines()]

    for name in file_names:

        # Source files
        image_src = os.path.join(images_folder, name + ".JPG")
        mask_src = os.path.join(masks_folder, name + ".PNG")

        # Save everything as PNG
        image_dst = os.path.join(img_output, name + ".png")
        mask_dst = os.path.join(mask_output, name + ".png")

        # Process image
        if os.path.exists(image_src):
            process_and_save(image_src, image_dst)
           # print(f"Saved Image: {image_dst}")
        else:
            print(f"Image Missing: {image_src}")

        # Process mask
        if os.path.exists(mask_src):
            process_and_save(mask_src, mask_dst)
            #print(f"Saved Mask: {mask_dst}")
        else:
            print(f"Mask Missing: {mask_src}")


# Process train dataset
copy_dataset(train_txt, train_images_out, train_masks_out)

# Process validation dataset
copy_dataset(val_txt, val_images_out, val_masks_out)

print("\nDataset preprocessing completed successfully!")


Dataset preprocessing completed successfully!


In [3]:
import os
import shutil

# =========================
# USER INPUT PATHS
# =========================

# Source dataset path
source_path = r"/mnt/lustre/lalit47/us_data/bench/OTU_2d"

# Target output path
target_path = r"/mnt/lustre/lalit47/down/class/MMOTU"

# Source images folder
images_folder = os.path.join(source_path, "images")

# Text files
train_cls_txt = os.path.join(source_path, "train_cls.txt")
val_cls_txt = os.path.join(source_path, "val_cls.txt")


def make_classification_data(txt_file, split_name):
    """
    txt_file format:
    image_name class_id

    Example:
    658.JPG 5
    384.JPG 3
    """

    with open(txt_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        if line == "":
            continue

        image_name, class_id = line.split()

        # Source image path
        src_image_path = os.path.join(images_folder, image_name)

        # Target class folder
        class_folder = os.path.join(target_path, split_name, class_id)
        os.makedirs(class_folder, exist_ok=True)

        # Target image path
        dst_image_path = os.path.join(class_folder, image_name)

        # Copy image
        if os.path.exists(src_image_path):
            shutil.copy(src_image_path, dst_image_path)
            #print(f"Copied: {image_name} → {split_name}/{class_id}")
        else:
            print(f"Image missing: {src_image_path}")


# Create train classification data
make_classification_data(train_cls_txt, "train")

# Create validation classification data
make_classification_data(val_cls_txt, "val")

print("\nClassification dataset created successfully!")


Classification dataset created successfully!


In [11]:
import os
import cv2

# =========================
# USER INPUT PATHS
# =========================

# Source dataset path
source_path = r"/mnt/lustre/lalit47/us_data/bench/OTU_2d"

# Target output path
target_path = r"/mnt/lustre/lalit47/down/class/MMOTU"

# Source images folder
images_folder = os.path.join(source_path, "images")

# Text files
train_cls_txt = os.path.join(source_path, "train_cls.txt")
val_cls_txt = os.path.join(source_path, "val_cls.txt")


def process_and_save(src, dst):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return

    # Resize image
    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    # Save image
    cv2.imwrite(dst, im)


def make_classification_data(txt_file, split_name):
    """
    txt_file format:
    image_name class_id

    Example:
    658.JPG 5
    384.JPG 3
    """

    with open(txt_file, "r") as f:
        lines = f.readlines()

    for line in lines:

        line = line.strip()

        if line == "":
            continue

        image_name, class_id = line.split()

        # Source image path
        src_image_path = os.path.join(images_folder, image_name)

        # Create class folder
        class_folder = os.path.join(target_path, split_name, class_id)
        os.makedirs(class_folder, exist_ok=True)

        # Convert filename to PNG
        base_name = os.path.splitext(image_name)[0]
        dst_image_name = base_name + ".png"

        # Target image path
        dst_image_path = os.path.join(class_folder, dst_image_name)

        # Process and save image
        if os.path.exists(src_image_path):
            process_and_save(src_image_path, dst_image_path)
            # print(f"Saved: {dst_image_path}")
        else:
            print(f"Image missing: {src_image_path}")


# Create train classification data
make_classification_data(train_cls_txt, "train")

# Create validation classification data
make_classification_data(val_cls_txt, "val")

print("\nClassification dataset created successfully!")


Classification dataset created successfully!
